In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:
### Initializing chroma db client
import chromadb

chroma_db_path = '/content/drive/MyDrive/data_movie/data_movie/plotDB_2'

client = chromadb.PersistentClient(path=chroma_db_path)

In [ ]:
import os

persist_dir_cache = "/content/drive/MyDrive/data_movie/data_movie/plot_cache"

os.makedirs(persist_dir_cache, exist_ok=True)

In [ ]:
client_cache = chromadb.PersistentClient(path=persist_dir_cache)

In [ ]:
import pandas as pd

from sentence_transformers import CrossEncoder, util

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def get_recommendation(query):

  collection = client.get_or_create_collection(name='Movie_plot')
  cache_collection_name = 'Cache_2'
  cache_collection = client_cache.get_or_create_collection(name=cache_collection_name)

  threshold = 0.2

  ids = []
  documents = []
  distances = []
  metadatas = []
  results_df = pd.DataFrame()

  cache_results = cache_collection.query(
    query_texts = [query],
    n_results = 2
  )

  # Check if the distance is greater than the threshold, if so, return results from the main collection
  if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
      # Query the collection against the user query and return the results
      results = collection.query(
          query_texts=query,
          n_results=5
      )

      # Store the query in cache_collection as a document with respect to ChromaDB for future reference
      # Store retrieved text, ids, distances, and metadatas in cache_collection as metadatas, so they can be fetched easily if a query indeed matches to a query in cache
      Keys = []
      Values = []

      for key, val in results.items():
          if val is None:
              continue
          for i in range(len(val[0])):  # Iterate over the actual length of val
              Keys.append(str(key) + str(i))
              if len(val[0]) > i:  # Check if the current index exists in val
                  Values.append(str(val[0][i]))

      cache_collection.add(
          documents=[query],
          ids=[query],
          metadatas=dict(zip(Keys, Values))
      )

      # Print message indicating the results are found in the main collection
      print("Not found in cache. Found in the main collection.")

      # Construct a DataFrame from the query results
      result_dict = {'Metadatas': results['metadatas'][0], 'Documents': results['documents'][0], 'Distances': results['distances'][0], "IDs": results["ids"][0]}
      results_df = pd.DataFrame.from_dict(result_dict)


  # If the distance is less than the threshold, return results from the cache
  elif cache_results['distances'][0][0] <= threshold and cache_results['ids']:
      cache_result_dict = cache_results['metadatas'][0][0]

      # Loop through each inner list and then through the dictionary
      for key, value in cache_result_dict.items():
          if 'ids' in key:
              ids.append(value)
          elif 'documents' in key:
              documents.append(value)
          elif 'distances' in key:
              distances.append(value)
          elif 'metadatas' in key:
              metadatas.append(value)

      # Print message indicating the results are found in the cache
      print("Found in cache!")

      # Create a DataFrame from the cached results
      results_df = pd.DataFrame({
          'IDs': ids,
          'Documents': documents,
          'Distances': distances,
          'Metadatas': metadatas
      })
  else:
      # Print message indicating no valid results found in cache
      print("No valid results found in cache!")
  print(results_df)

  cross_inputs = [[query, response] for response in results_df['Documents']]
  cross_rerank_scores = cross_encoder.predict(cross_inputs)

  results_df['Reranked_scores'] = cross_rerank_scores

  rank = results_df.sort_values(by='Reranked_scores')

  return rank.head(5)



In [ ]:
#### filter creator

import re

def clean_text(desc):
  modified_text = desc.replace("<ul>", "").replace("</ul>", "").replace("<li>", "").replace("</li>", ",").replace("<br>", ".")
  final = re.sub(r'<[^>]*>', '', modified_text).replace(". .",".")
  return final

def truncate_text(text, max_words=200):
    return " ".join(text.split()[:max_words])


import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_movie_plot(movie_name):
    base_url = "https://en.wikipedia.org/w/api.php"

    # Step 1: Search correct page
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": movie_name + " film",
        "format": "json"
    }

    res = requests.get(base_url, params=search_params, headers=HEADERS)
    data = res.json()

    results = data.get("query", {}).get("search", [])
    if not results:
        return "Movie not found"

    page_title = results[0]["title"]

    # Step 2: Get sections
    section_params = {
        "action": "parse",
        "page": page_title,
        "prop": "sections",
        "format": "json"
    }

    res = requests.get(base_url, params=section_params, headers=HEADERS)
    data = res.json()

    sections = data.get("parse", {}).get("sections", [])

    # Step 3: Find plot section index
    plot_index = None
    for sec in sections:
        title = sec["line"].lower()
        if any(k in title for k in ["plot", "synopsis", "premise"]):
            plot_index = sec["index"]
            break

    if not plot_index:
        return "Plot section not found"

    # Step 4: Fetch that section content
    content_params = {
        "action": "parse",
        "page": page_title,
        "prop": "text",
        "section": plot_index,
        "format": "json"
    }

    res = requests.get(base_url, params=content_params, headers=HEADERS)
    data = res.json()

    html = data["parse"]["text"]["*"]

    # Step 5: Clean HTML → text
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")

    paragraphs = [p.get_text() for p in soup.find_all("p")]

    return "\n".join(paragraphs).strip()

def create_query(movie_name):

  movie_plot = get_movie_plot(movie_name)
  short_plot = truncate_text(clean_text(movie_plot))

  query = f"""

  Find movie plots similar to:

  {short_plot}

  """

  return query





In [ ]:
query = create_query("The Shawshank Redemption")
get_recommendation(query)

Not found in cache. Found in the main collection.
                                           Metadatas  \
0  {'Release_Year': 1994, 'short_plot': 'In 1947 ...   
1  {'Release_Year': 1993, 'short_plot': 'Andy and...   
2  {'Title': 'Human Experiments', 'Release_Year':...   
3  {'Title': 'The Big Night', 'Release_Year': 195...   
4  {'Title': 'Harry Brown', 'short_plot': 'Harry ...   

                                           Documents  Distances    IDs  
0  In 1947 Portland, Maine, banker Andy Dufresne ...   0.140234   7022  
1  Andy and Tracy Safian are a newlywed couple li...   0.847172   6742  
2  Rachel Foster (Linda Haynes) is a country sing...   0.907499   4323  
3  On his teenaged son Georgie's birthday, Andy L...   0.909854     22  
4  Harry Brown is an elderly pensioner who was on...   0.929087  14830  


,Metadatas,Documents,Distances,IDs,Reranked_scores
4,"{'Title': 'Harry Brown', 'short_plot': 'Harry ...",Harry Brown is an elderly pensioner who was on...,0.929087,14830,-7.292022
3,"{'Title': 'The Big Night', 'Release_Year': 195...","On his teenaged son Georgie's birthday, Andy L...",0.909854,22,-6.491680
2,"{'Title': 'Human Experiments', 'Release_Year':...",Rachel Foster (Linda Haynes) is a country sing...,0.907499,4323,-6.444994
1,"{'Release_Year': 1993, 'short_plot': 'Andy and...",Andy and Tracy Safian are a newlywed couple li...,0.847172,6742,-5.343904
0,"{'Release_Year': 1994, 'short_plot': 'In 1947 ...","In 1947 Portland, Maine, banker Andy Dufresne ...",0.140234,7022,3.218917


In [ ]:
query

'\n\n  Find movie plots similar to:\n\n  In 1947, Portland, Maine, banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd "Red" Redding, a contraband smuggler serving a life sentence, who procures for him a rock hammer and a large poster of Rita Hayworth. Assigned to work in the prison laundry, Andy is frequently raped by "the Sisters" gang, led by Bogs Diamond. In 1949, Andy overhears the captain of the guards, Byron Hadley, complaining about being taxed on an inheritance and offers to help him shelter the money legally. After the Sisters beat Andy to near-death, Hadley cripples Bogs, who is subsequently transferred to a minimum-security hospital; Andy is not attacked again. Warden Samuel Norton assigns Andy to the prison\'s decrepit library, ostensibly to assist elderly inmate Brooks Hatlen, but in reality, to use Andy\'s financial expertise in managing the warden\'s and other